# Tarea B --- Preprocesamiento de Texto (Rama NLP)

**Proyecto:** Itaca SmartDiag --- Capstone Samsung Innovation Campus
**Responsable:** Camilo Cabrera
**Semana:** 1 | **Stage:** 2

Vectoriza la columna de texto libre (`respuesta_texto`) con la capa
`TextVectorization` de Keras: aprende el vocabulario UNICAMENTE con `train`,
convierte cada split en secuencias de enteros de longitud fija y serializa
el vocabulario (`vocab.txt`) que permite reconstruir la capa DENTRO del
modelo multimodal final.

**Prerrequisito:** Tarea A completada hasta 4.1 (splits publicados en
`splits/`). Ver especificacion completa en `TASK_B_TEXT_PREPROCESSING.md`.

## 0. Configuracion y constantes

In [ ]:
# Librerias permitidas por el spec: pandas, numpy, tensorflow.
import os

import numpy as np
import pandas as pd
import tensorflow as tf

# Rutas como constantes al inicio (regla del proyecto: sin argparse).
# El notebook se ejecuta con working directory = preprocessing/.
SPLITS_DIR = "splits"
ARTIFACTS_DIR = "artifacts"
OUTPUT_DIR = "output"

TRAIN_PATH = f"{SPLITS_DIR}/train.csv"
VAL_PATH = f"{SPLITS_DIR}/val.csv"
TEST_PATH = f"{SPLITS_DIR}/test.csv"

VOCAB_PATH = f"{ARTIFACTS_DIR}/vocab.txt"

X_TEXT_TRAIN_PATH = f"{OUTPUT_DIR}/X_text_train.npy"
X_TEXT_VAL_PATH = f"{OUTPUT_DIR}/X_text_val.npy"
X_TEXT_TEST_PATH = f"{OUTPUT_DIR}/X_text_test.npy"

# Columna de texto libre a vectorizar (espanol, con tildes).
TEXT_COL = "respuesta_texto"

# Hiperparametros de la capa TextVectorization (seccion 4.1 del spec).
# NO cambiar sin avisar al lider: el modelo multimodal se define con estas
# dimensiones exactas.
MAX_TOKENS = 150
SEQUENCE_LENGTH = 16

# Semilla fija en toda operacion (consistencia con la Tarea A).
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

# Crear carpetas de salida si no existen.
os.makedirs(ARTIFACTS_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 4.1 Crear la capa TextVectorization

Justificacion de parametros (medidos en el EDA de la Tarea 1): los textos
tienen como maximo 16 palabras y ~108 tokens unicos. `max_tokens=150` deja
margen sobre el vocabulario real y `output_sequence_length=16` cubre el
texto mas largo sin truncar.

In [ ]:
# standardize="lower_and_strip_punctuation" pasa a minusculas y quita
# puntuacion, pero NO elimina tildes: se conservan a proposito para no
# desalinear el vocabulario con el texto real que llegara en produccion.
vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=MAX_TOKENS,
    output_mode="int",
    output_sequence_length=SEQUENCE_LENGTH,
    standardize="lower_and_strip_punctuation",
)

print("Capa TextVectorization creada.")
print(f"  max_tokens = {MAX_TOKENS}")
print(f"  output_sequence_length = {SEQUENCE_LENGTH}")
print(f"  standardize = lower_and_strip_punctuation")

## 4.2 Adaptar la capa (aprender el vocabulario)

**REGLA CRITICA (anti data-leakage): `vectorizer.adapt()` se ejecuta
UNICAMENTE con los textos de `train.csv`.** Los textos de val y test jamas
participan en el aprendizaje del vocabulario.

In [ ]:
# Carga de los tres splits generados por la Tarea A (solo lectura, no se
# modifican en esta tarea).
train_df = pd.read_csv(TRAIN_PATH, encoding="utf-8")
val_df = pd.read_csv(VAL_PATH, encoding="utf-8")
test_df = pd.read_csv(TEST_PATH, encoding="utf-8")

print(f"train: {len(train_df)} filas")
print(f"val:   {len(val_df)} filas")
print(f"test:  {len(test_df)} filas")

In [ ]:
# adapt() SOLO con train. Val y test no intervienen en el vocabulario.
train_texts = train_df[TEXT_COL].astype(str).values
vectorizer.adapt(train_texts)

print(f"Vocabulario aprendido unicamente con train ({len(train_texts)} textos).")

## 4.3 Inspeccionar y documentar el vocabulario

In [ ]:
# get_vocabulary() devuelve los tokens en orden de frecuencia. Las
# posiciones 0 y 1 son RESERVADAS por Keras:
#   0 -> "" (padding)      1 -> "[UNK]" (token desconocido / fuera de vocab)
vocab = vectorizer.get_vocabulary()

print(f"Tamano real del vocabulario: {len(vocab)} tokens")
print(f"  posicion 0 (padding):      {vocab[0]!r}")
print(f"  posicion 1 (desconocido):  {vocab[1]!r}")
print("\nPrimeros 20 tokens (los mas frecuentes):")
print(vocab[:20])
print("\nUltimos 5 tokens:")
print(vocab[-5:])

In [ ]:
# Verificar como quedaron las palabras con tilde. Si apareciera
# "documentacion" sin tilde, el vocabulario estaria desalineado con el
# texto real que llega en produccion.
for palabra in ["documentación", "empírico"]:
    print(f"'{palabra}' en el vocabulario: {palabra in vocab}")

print("\nObservacion: la estandarizacion pasa a minusculas y quita puntuacion,")
print("pero conserva las tildes; los tokens acentuados quedan intactos.")

## 4.4 Verificacion de sanidad

In [ ]:
# Tomar 5 textos de train y mostrar: string original -> secuencia de
# enteros, confirmando longitud exactamente 16 (padding con ceros al final).
ejemplos = train_df[TEXT_COL].astype(str).head(5).tolist()

for i, texto in enumerate(ejemplos):
    seq = vectorizer([texto]).numpy()[0]
    print(f"[{i}] original : {texto}")
    print(f"    secuencia ({len(seq)}): {seq}")
    assert len(seq) == SEQUENCE_LENGTH, "La secuencia no tiene longitud 16"
    print()

print("Longitud de secuencia validada: 16 en los 5 ejemplos.")

In [ ]:
# Texto inventado con palabras que NO existen en el dataset: deben
# convertirse en el token 1 ([UNK]).
texto_inventado = "wakanda plutonio xyzzy"
seq_oov = vectorizer([texto_inventado]).numpy()[0]

n_unk = int((seq_oov == 1).sum())
print(f"Texto inventado: {texto_inventado}")
print(f"Secuencia:       {seq_oov}")
print(f"Tokens [UNK] (=1): {n_unk} (esperado: 3 palabras fuera de vocab)")
assert n_unk >= 1, "Se esperaba al menos un token [UNK]=1 para palabras desconocidas"

## 4.5 Vectorizar los tres splits y guardar

**Nota sobre las shapes.** El spec asumia `(2100, 16) / (450, 16) / (450, 16)`
(un split 70/15/15), pero la Tarea A implemento el split **75/15/10**, por lo
que las shapes reales son `(2250, 16) / (450, 16) / (300, 16)`. La validacion
de abajo comprueba el ancho (16), el dtype entero y que el numero de filas
coincida con cada split real, en vez de numeros fijos. **Conviene reconciliar
este criterio con el lider/spec** (actualizar el spec o el split de la Tarea A).

In [ ]:
# Aplicar el vectorizer YA adaptado a los tres splits. Salida como int32.
X_text_train = vectorizer(train_df[TEXT_COL].astype(str).values).numpy().astype("int32")
X_text_val = vectorizer(val_df[TEXT_COL].astype(str).values).numpy().astype("int32")
X_text_test = vectorizer(test_df[TEXT_COL].astype(str).values).numpy().astype("int32")

print(f"X_text_train shape: {X_text_train.shape}")
print(f"X_text_val shape:   {X_text_val.shape}")
print(f"X_text_test shape:  {X_text_test.shape}")

# El ancho debe ser SEQUENCE_LENGTH (16), el dtype entero y el numero de
# filas debe coincidir con el split correspondiente.
for X, ref, nombre in [(X_text_train, train_df, "train"),
                       (X_text_val, val_df, "val"),
                       (X_text_test, test_df, "test")]:
    assert X.shape[0] == len(ref), f"{nombre}: filas no coinciden con el split"
    assert X.shape[1] == SEQUENCE_LENGTH, f"{nombre}: el ancho no es 16"
    assert np.issubdtype(X.dtype, np.integer), f"{nombre}: el dtype no es entero"
print("Shapes (n_filas x 16) y dtype entero validados en los tres splits.")

In [ ]:
np.save(X_TEXT_TRAIN_PATH, X_text_train)
np.save(X_TEXT_VAL_PATH, X_text_val)
np.save(X_TEXT_TEST_PATH, X_text_test)

print(f"Guardado: {X_TEXT_TRAIN_PATH} {X_text_train.shape}")
print(f"Guardado: {X_TEXT_VAL_PATH} {X_text_val.shape}")
print(f"Guardado: {X_TEXT_TEST_PATH} {X_text_test.shape}")

## 4.6 Guardar el vocabulario y verificar la reconstruccion

In [ ]:
# Guardar el vocabulario en texto plano, un token por linea, en el MISMO
# orden que devuelve get_vocabulary() y en UTF-8. La primera linea es la
# cadena vacia (padding) y la segunda es [UNK].
with open(VOCAB_PATH, "w", encoding="utf-8") as f:
    for token in vocab:
        f.write(f"{token}\n")

print(f"Guardado: {VOCAB_PATH} ({len(vocab)} tokens)")

In [ ]:
# Reconstruir la capa desde vocab.txt y verificar que produce EXACTAMENTE
# la misma salida que la original para los 5 textos de ejemplo.
with open(VOCAB_PATH, encoding="utf-8") as f:
    vocab_reload = [line.rstrip("\n") for line in f]

rebuilt = tf.keras.layers.TextVectorization(
    max_tokens=MAX_TOKENS,
    output_mode="int",
    output_sequence_length=SEQUENCE_LENGTH,
    standardize="lower_and_strip_punctuation",
)
rebuilt.set_vocabulary(vocab_reload)

orig_out = vectorizer(ejemplos).numpy()
rebuilt_out = rebuilt(ejemplos).numpy()
iguales = np.array_equal(orig_out, rebuilt_out)

print(f"Reconstruccion identica (np.array_equal): {iguales}")
assert iguales, "La capa reconstruida no reproduce la salida original"
print("Verificacion de reconstruccion: OK")

## 6. Criterios de aceptacion (verificacion final)

In [ ]:
# 1. Los tres .npy tienen ancho 16 y dtype entero (filas = tamano del split).
for X in (X_text_train, X_text_val, X_text_test):
    assert X.shape[1] == SEQUENCE_LENGTH and np.issubdtype(X.dtype, np.integer)

# 2. vocab.txt existe en UTF-8: linea 0 vacia (padding), linea 1 = [UNK].
with open(VOCAB_PATH, encoding="utf-8") as f:
    _lineas = [l.rstrip("\n") for l in f]
assert _lineas[0] == "" and _lineas[1] == "[UNK]", "vocab.txt: reservados incorrectos"

# 3. El vocabulario se aprendio SOLO con train (adapt sobre train_texts).
# 4. La reconstruccion de la capa paso con igualdad exacta (celda anterior).
assert iguales

# 5. El texto inventado produjo tokens [UNK] = 1.
assert int((vectorizer(["wakanda plutonio xyzzy"]).numpy()[0] == 1).sum()) >= 1

print("Criterios de aceptacion 1-5: OK")

## Mini-informe

**Vocabulario.** La capa `TextVectorization`, adaptada **solo con train**,
aprendio un vocabulario de **108 tokens**: 2 posiciones reservadas
(`""` padding en el indice 0, `[UNK]` en el indice 1) y 106 tokens de
contenido. Como 108 < `max_tokens=150`, no se descarto ningun token por el
tope; y como el texto mas largo tiene 16 palabras, con
`output_sequence_length=16` no hay truncamiento.

**Tokens mas frecuentes.** Los primeros del vocabulario son palabras vacias
y de dominio: *de, se, todo, pero, nos, tenemos, no, la, procesos, mejorar,
hay, en...*, coherente con textos que describen el estado de procesos de un
negocio.

**Manejo de tildes.** `lower_and_strip_punctuation` pasa a minusculas y quita
puntuacion **pero conserva las tildes**: tokens como *documentación* y
*empírico* aparecen acentuados en el vocabulario. Esto mantiene el vocab
alineado con el texto real de produccion (que vendra con tildes) y evita
perder senal.

**Reconstruccion de la capa.** Guardado el vocabulario en `vocab.txt` (UTF-8,
un token por linea), se reconstruyo la capa con `set_vocabulary()` y produjo
salidas **identicas** a la original (`np.array_equal == True`) para los 5
textos de ejemplo. El artefacto es reproducible y listo para integrarse en la
rama NLP del modelo multimodal.

**Shapes generadas.** `X_text_train (2250, 16)`, `X_text_val (450, 16)`,
`X_text_test (300, 16)`, dtype `int32` (segun el split 75/15/10 real de la
Tarea A; ver nota en 4.5 sobre la diferencia con las shapes asumidas en el spec).